# TejaLens — Module 4 Training
**Before running:** Runtime → Change runtime type → T4 GPU → Save

This notebook downloads datasets directly from source (no manual upload needed).

In [ ]:
# Cell 1 — Install dependencies
!pip install timm opencv-python scikit-learn matplotlib tqdm -q
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT AVAILABLE — check runtime type')
print('torch:', torch.__version__)

In [ ]:
# Cell 2 — Clone project from GitHub (push your code to GitHub first)
# OR manually upload the src/ folder using the Files panel on the left

# Option A: GitHub (recommended)
!git clone https://github.com/YOUR_USERNAME/tejalens-skin-prescreen.git
%cd tejalens-skin-prescreen

# Option B: Upload src/ manually via Files panel, then:
# import os; os.chdir('/content')  # adjust if needed

In [ ]:
# Cell 3 — Download HAM10000 from Harvard Dataverse (no auth required)
import zipfile, pathlib, requests
from tqdm import tqdm

DEST = pathlib.Path('data/ham10000')
DEST.mkdir(parents=True, exist_ok=True)

FILES = [
    {'id': 4338392, 'name': 'HAM10000_metadata.tab', 'zip': False},
    {'id': 3172585, 'name': 'HAM10000_images_part_1.zip', 'zip': True},
    {'id': 3172584, 'name': 'HAM10000_images_part_2.zip', 'zip': True},
]

BASE = 'https://dataverse.harvard.edu/api/access/datafile'
HEADERS = {'User-Agent': 'Mozilla/5.0'}

for f in FILES:
    dest_path = DEST / f['name']
    if dest_path.exists():
        print(f"Already exists: {f['name']}")
        continue
    print(f"Downloading {f['name']}...")
    r = requests.get(f"{BASE}/{f['id']}", headers=HEADERS, stream=True, timeout=300)
    total = int(r.headers.get('content-length', 0))
    with open(dest_path, 'wb') as out, tqdm(total=total, unit='B', unit_scale=True) as bar:
        for chunk in r.iter_content(65536):
            out.write(chunk)
            bar.update(len(chunk))
    if f['zip']:
        print(f"Unzipping {f['name']}...")
        with zipfile.ZipFile(dest_path) as z:
            z.extractall(DEST)
        dest_path.unlink()

print('HAM10000 images:', len(list(DEST.glob('*.jpg'))))

In [ ]:
# Cell 4 — Download ISIC 2018 Task 1 (segmentation) + ISIC 2019
import zipfile, pathlib, requests
from tqdm import tqdm

BASE = 'https://isic-challenge-data.s3.amazonaws.com'
DOWNLOADS = [
    {'name': 'ISIC2018 Training Images', 'url': f'{BASE}/2018/ISIC2018_Task1-2_Training_Input.zip',       'dest': pathlib.Path('data/isic2018_seg'), 'unzip': True},
    {'name': 'ISIC2018 Seg Masks',       'url': f'{BASE}/2018/ISIC2018_Task1_Training_GroundTruth.zip',   'dest': pathlib.Path('data/isic2018_seg'), 'unzip': True},
    {'name': 'ISIC2019 Training Images', 'url': f'{BASE}/2019/ISIC_2019_Training_Input.zip',              'dest': pathlib.Path('data/isic2019'),     'unzip': True},
    {'name': 'ISIC2019 Ground Truth',    'url': f'{BASE}/2019/ISIC_2019_Training_GroundTruth.csv',        'dest': pathlib.Path('data/isic2019'),     'unzip': False},
]

for entry in DOWNLOADS:
    entry['dest'].mkdir(parents=True, exist_ok=True)
    fname = entry['url'].split('/')[-1]
    out_path = entry['dest'] / fname
    if out_path.exists() or (entry['unzip'] and (entry['dest'] / out_path.stem).exists()):
        print(f'Already done: {fname}')
        continue
    print(f"Downloading {entry['name']}...")
    r = requests.get(entry['url'], stream=True, timeout=300)
    total = int(r.headers.get('content-length', 0))
    with open(out_path, 'wb') as f, tqdm(total=total, unit='B', unit_scale=True, desc=fname[:40]) as bar:
        for chunk in r.iter_content(65536):
            f.write(chunk)
            bar.update(len(chunk))
    if entry['unzip']:
        with zipfile.ZipFile(out_path) as z:
            z.extractall(entry['dest'])
        out_path.unlink()
        print(f'  Extracted -> {entry["dest"]}')

print('ISIC2018 images:', len(list(pathlib.Path('data/isic2018_seg/ISIC2018_Task1-2_Training_Input').glob('*.jpg'))))
print('ISIC2018 masks: ', len(list(pathlib.Path('data/isic2018_seg/ISIC2018_Task1_Training_GroundTruth').glob('*.png'))))
print('ISIC2019 images:', len(list(pathlib.Path('data/isic2019/ISIC_2019_Training_Input').glob('*.jpg'))))

In [ ]:
# Cell 5 — Train segmentation: U-Net (VGG16) on ISIC 2018 Task 1
from src.segmentation.train import train as train_seg

train_seg(
    encoder='vgg16',
    epochs=30,
    lr=1e-4,
    image_size=256,
    batch_size=8,
    save_path='unet_vgg16.pth'
)
# Target: val_dice > 0.90

In [ ]:
# Cell 6 — Train research classifier: Swin-Small on HAM10000
from src.classification.train import train as train_cls

_, swin_metrics = train_cls(
    model_name='swin_small',
    dataset_name='ham10000',
    epochs=30,
    lr=1e-4,
    batch_size=32,
    loss_type='focal',
    save_path='swin_small_ham10000.pth'
)
print('Swin-Small final metrics:', swin_metrics)

In [ ]:
# Cell 7 — Train on-device classifier: EfficientNet-B0 on HAM10000
from src.classification.train import train as train_cls

_, b0_metrics = train_cls(
    model_name='efficientnet_b0',
    dataset_name='ham10000',
    epochs=30,
    lr=1e-4,
    batch_size=32,
    loss_type='focal',
    save_path='efficientnet_b0_ham10000.pth'
)
print('EfficientNet-B0 final metrics:', b0_metrics)

In [ ]:
# Cell 8 — Fine-tune EfficientNet-B0 on ISIC 2019 (8 classes)
from src.classification.train import train as train_cls

_, b0_19_metrics = train_cls(
    model_name='efficientnet_b0',
    dataset_name='isic2019',
    epochs=20,
    lr=5e-5,
    batch_size=32,
    loss_type='focal',
    save_path='efficientnet_b0_isic2019.pth'
)
print('EfficientNet-B0 ISIC2019 metrics:', b0_19_metrics)

In [ ]:
# Cell 9 — Save results to reports/
import json, pathlib
pathlib.Path('reports').mkdir(exist_ok=True)

# Fill in seg metrics from Cell 5 output
results = {
    'unet_vgg16_isic2018': {
        'pixel_accuracy': None,   # <-- fill in from Cell 5 output
        'jaccard': None,
        'dice': None,
    },
    'swin_small_ham10000': swin_metrics,
    'efficientnet_b0_ham10000': b0_metrics,
    'efficientnet_b0_isic2019': b0_19_metrics,
}

with open('reports/module4_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print(json.dumps(results, indent=2))

In [ ]:
# Cell 10 — Download trained weights and results
from google.colab import files
for fname in ['unet_vgg16.pth', 'swin_small_ham10000.pth', 'efficientnet_b0_ham10000.pth', 'efficientnet_b0_isic2019.pth', 'reports/module4_results.json']:
    try:
        files.download(fname)
        print(f'Downloaded: {fname}')
    except Exception as e:
        print(f'Could not download {fname}: {e}')